In [3]:
from pyspark.ml.feature import VectorAssembler

# example: a new, hypothetical disruption a Transport Authority might enter
new_disruption = spark.createDataFrame([
    ("roadworks", "true", "14:30:00", "Wednesday"),
], ["reason", "planned", "departure_time", "weekday_name"])

# apply the same feature engineering used during training
new_disruption = new_disruption.withColumn(
    "departure_hour", F.split(F.col("departure_time"), ":")[0].cast("int")
).withColumn(
    "hour_sin", F.sin(2 * 3.14159265 * F.col("departure_hour") / 24)
).withColumn(
    "hour_cos", F.cos(2 * 3.14159265 * F.col("departure_hour") / 24)
).withColumn(
    "is_weekend", F.when(F.col("weekday_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)

# apply the saved encoding pipeline
encoded = pipeline_model.transform(new_disruption)

# VectorAssembler was applied as a separate step during training (not saved
# as part of the pipeline), so we reapply it here with the same feature columns
feature_cols = ["hour_sin", "hour_cos", "is_weekend", "reason_ohe", "planned_ohe"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
encoded_with_features = assembler.transform(encoded)

prediction = rf_model.transform(encoded_with_features)
prediction.select("reason", "planned", "weekday_name", "departure_time", "prediction", "probability").show(truncate=False)

26/07/15 17:18:32 WARN StringIndexerModel: Input column severity does not exist during transformation. Skip StringIndexerModel for this column.
                                                                                

+---------+-------+------------+--------------+----------+---------------------------------------------------------------+
|reason   |planned|weekday_name|departure_time|prediction|probability                                                    |
+---------+-------+------------+--------------+----------+---------------------------------------------------------------+
|roadworks|true   |Wednesday   |14:30:00      |1.0       |[0.3903691418659173,0.5302175620724641,0.07941329606161857,0.0]|
+---------+-------+------------+--------------+----------+---------------------------------------------------------------+



In [4]:
severity_labels = {0.0: "severe", 1.0: "moderate", 2.0: "minor"}

result = prediction.select("reason", "planned", "weekday_name", "departure_time", "prediction", "probability").collect()[0]
predicted_severity = severity_labels[result["prediction"]]

print(f"Input disruption: {result['reason']}, planned={result['planned']}, {result['weekday_name']} at {result['departure_time']}")
print(f"Predicted severity: {predicted_severity}")
print(f"Confidence: {max(result['probability']):.1%}")

Input disruption: roadworks, planned=true, Wednesday at 14:30:00
Predicted severity: moderate
Confidence: 53.0%


In [5]:
new_disruptions = spark.createDataFrame([
    ("roadworks", "true", "14:30:00", "Wednesday"),
    ("vandalism", "false", "22:00:00", "Saturday"),
    ("roadClosed", "true", "08:00:00", "Monday"),
], ["reason", "planned", "departure_time", "weekday_name"])

new_disruptions = new_disruptions.withColumn(
    "departure_hour", F.split(F.col("departure_time"), ":")[0].cast("int")
).withColumn(
    "hour_sin", F.sin(2 * 3.14159265 * F.col("departure_hour") / 24)
).withColumn(
    "hour_cos", F.cos(2 * 3.14159265 * F.col("departure_hour") / 24)
).withColumn(
    "is_weekend", F.when(F.col("weekday_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)

encoded_multi = pipeline_model.transform(new_disruptions)
encoded_multi_with_features = assembler.transform(encoded_multi)
predictions_multi = rf_model.transform(encoded_multi_with_features)

predictions_multi.select("reason", "planned", "weekday_name", "departure_time", "prediction").show(truncate=False)

26/07/15 17:19:30 WARN StringIndexerModel: Input column severity does not exist during transformation. Skip StringIndexerModel for this column.


+----------+-------+------------+--------------+----------+
|reason    |planned|weekday_name|departure_time|prediction|
+----------+-------+------------+--------------+----------+
|roadworks |true   |Wednesday   |14:30:00      |1.0       |
|vandalism |false  |Saturday    |22:00:00      |1.0       |
|roadClosed|true   |Monday      |08:00:00      |0.0       |
+----------+-------+------------+--------------+----------+



In [6]:
def predict_severity(reason, planned, weekday_name, departure_time):
    """Takes disruption details and returns the predicted severity."""
    row = spark.createDataFrame([
        (reason, planned, departure_time, weekday_name)
    ], ["reason", "planned", "departure_time", "weekday_name"])

    row = row.withColumn(
        "departure_hour", F.split(F.col("departure_time"), ":")[0].cast("int")
    ).withColumn(
        "hour_sin", F.sin(2 * 3.14159265 * F.col("departure_hour") / 24)
    ).withColumn(
        "hour_cos", F.cos(2 * 3.14159265 * F.col("departure_hour") / 24)
    ).withColumn(
        "is_weekend", F.when(F.col("weekday_name").isin("Saturday", "Sunday"), 1).otherwise(0)
    )

    encoded = pipeline_model.transform(row)
    encoded_with_features = assembler.transform(encoded)
    result = rf_model.transform(encoded_with_features).collect()[0]

    severity_labels = {0.0: "severe", 1.0: "moderate", 2.0: "minor"}
    predicted = severity_labels[result["prediction"]]
    confidence = max(result["probability"])

    return predicted, confidence

In [7]:
print("Bus Disruption Severity Predictor")
print("-" * 40)

reason = input("Reason (roadworks / roadClosed / routeDiversion / vandalism): ").strip()
planned = input("Planned? (true / false): ").strip().lower()
weekday_name = input("Day of week (e.g. Monday): ").strip().capitalize()
departure_time = input("Departure time (HH:MM:SS, e.g. 14:30:00): ").strip()

predicted, confidence = predict_severity(reason, planned, weekday_name, departure_time)

print(f"\nPredicted severity: {predicted.upper()}")
print(f"Model confidence: {confidence:.1%}")

Bus Disruption Severity Predictor
----------------------------------------


Reason (roadworks / roadClosed / routeDiversion / vandalism):  roadworks
Planned? (true / false):  false
Day of week (e.g. Monday):  Tuesday
Departure time (HH:MM:SS, e.g. 14:30:00):  12:30:00


26/07/15 17:33:18 WARN StringIndexerModel: Input column severity does not exist during transformation. Skip StringIndexerModel for this column.



Predicted severity: MODERATE
Model confidence: 55.9%
